# Week 4, Lab 3 — LangGraph state machine

Draft → critique → revise or stop.


In [ ]:
WEEK = 'Week 4'
LAB = 'Lab 3 — state machine'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate langchain langchain-huggingface langgraph
else:
    %pip install -q langchain langchain-ollama langgraph ollama


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

llm = get_langchain_llm()

class State(TypedDict):
    topic: str
    draft: str
    critique: str
    loops: int

def draft_node(state: State) -> State:
    prompt = f"Write 3 sentences for students on: {state['topic']}"
    if state.get("critique"):
        prompt += f"\nRevise using this critique: {state['critique']}"
    text = llm.invoke(prompt).content
    return {**state, "draft": text, "loops": state.get("loops", 0) + 1}

def critique_node(state: State) -> State:
    text = llm.invoke(
        f"Critique this student explainer in one sentence. If it is clear enough, reply with exactly APPROVED.\n\n{state['draft']}"
    ).content
    return {**state, "critique": text}

def should_continue(state: State) -> str:
    if "APPROVED" in (state.get("critique") or "").upper():
        return "stop"
    if state.get("loops", 0) >= 3:
        return "stop"
    return "revise"

g = StateGraph(State)
g.add_node("draft", draft_node)
g.add_node("critique", critique_node)
g.add_edge(START, "draft")
g.add_edge("draft", "critique")
g.add_conditional_edges("critique", should_continue, {"revise": "draft", "stop": END})
app = g.compile()
out = app.invoke({"topic": "ReAct agents", "draft": "", "critique": "", "loops": 0})
print("LOOPS", out["loops"])
print("CRITIQUE", out["critique"])
print("DRAFT\n", out["draft"])
print(app.get_graph().print_ascii())


Always cap loops. Then run `lab4_langgraph_multiagent.ipynb` (already in this folder).
